# AI Brochure Generator

## Importing the libraries

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from IPython.display import Markdown, display, update_display

## Loading OpenAI API Key

In [2]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


In [3]:
client = OpenAI()

## Fetching contents of website

In [4]:
def fetch_content(url):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.get(url)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        for element in soup(["script", "style", "nav", "footer", "header"]):
            element.decompose()

        clean_text = soup.get_text(separator = "\n")
        clean_text = " ".join([line.strip() for line in clean_text.splitlines() if line.strip()])
        clean_text = clean_text[:500]
        
        return clean_text

    finally:
        driver.quit()

## Fetching links on websites

In [5]:
def fetch_links(url):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.get(url)
        soup = BeautifulSoup(driver.page_source, "html.parser") 

        links = [link.get("href") for link in soup.find_all("a")]
        links = [link for link in links if link]

        return links

    finally:
        driver.quit()

## Calling Through OpenAI API

In [ ]:
def gpt_relevant_links(url):

    print(f"Selecting relevant links for {url} ...")

    response = client.responses.create(
        model = "gpt-5.4-mini",
        input = [
            {
                "role": "system",
                "content": link_system_prompt
            },

            {
                "role": "user",
                "content": link_user_prompt(url)
            }  
        ],
        text = { "format": { "type": "json_object" } }
    )

    result = response.output_text
    
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links!")

    return links

## Fetching links and contents

In [7]:
def fetch_links_content(url):

    content = fetch_content(url)
    links = gpt_relevant_links(url)

    result = f"## Landing Page: \n\n {content} \n\n ##Relevant Links: \n"

    for link in links["links"]:
        result += f"\n\n ### Link: {link['type']} \n"
        result += fetch_content(link["url"])

    return result

## Defining Prompts

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}   
"""

In [9]:
def link_user_prompt(url):

    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """

    links = fetch_links(url)
    user_prompt += "\n".join(links)

    return user_prompt

## Calling relevant links function

In [10]:
gpt_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co ...
Found 13 relevant links!


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand', 'url': 'https://huggingface.co/brand'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}]}

In [11]:
gpt_relevant_links("https://ollama.com/")

Selecting relevant links for https://ollama.com/ ...
Found 10 relevant links!


{'links': [{'type': 'homepage', 'url': 'https://ollama.com/'},
  {'type': 'docs', 'url': 'https://ollama.com/docs'},
  {'type': 'pricing', 'url': 'https://ollama.com/pricing'},
  {'type': 'download', 'url': 'https://ollama.com/download'},
  {'type': 'blog', 'url': 'https://ollama.com/blog'},
  {'type': 'docs', 'url': 'https://docs.ollama.com'},
  {'type': 'github', 'url': 'https://github.com/ollama/ollama'},
  {'type': 'community', 'url': 'https://discord.com/invite/ollama'},
  {'type': 'social', 'url': 'https://twitter.com/ollama'},
  {'type': 'careers page', 'url': 'https://jobs.ashbyhq.com/ollama'}]}

## Defining Brochure Prompts

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [13]:
def brochure_user_prompt(name, url):

    user_prompt = f"""
    You are looking at a company called: {name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """

    user_prompt += fetch_links_content(url)
    return user_prompt

## Creating brochure with streamed output

In [ ]:
def create_brochure(name, url):
    
    stream = client.responses.create(
        model = "gpt-5.4-mini",
        input = [
            {
                "role": "system",
                "content": brochure_system_prompt
            },

            {
                "role": "user",
                "content": brochure_user_prompt(name, url)
            }  
        ],
        stream = True
    )

    response = ""
    display_handle = display(
        Markdown(""), display_id = True
    )

    for event in stream:
        if event.type == "response.output_text.delta":
            response += event.delta
            update_display(
                Markdown(response),
                display_id = display_handle.display_id
            )


In [17]:
create_brochure("Hugging Face", "https://huggingface.co")

Selecting relevant links for https://huggingface.co ...
Found 15 relevant links!


# Hugging Face

Hugging Face is an AI and machine learning company best known as **the collaboration platform for the machine learning community**. It provides a central place where people can **share, explore, discover, and experiment with models, datasets, and applications**. The company’s mission is to help build **an open and ethical AI future together**.

## What they do

Hugging Face offers a broad AI ecosystem, including:

- **The Hugging Face Hub** for hosting Git-based models, datasets, and Spaces
- **Python and JavaScript client libraries** for working with Hugging Face services
- **Tools for agents and developers** through CLI and SDKs
- **Task exploration tools** for demos, models, and datasets across ML use cases
- **Deployment and inference services**
- **Dataset viewer and metadata APIs**
- **Enterprise support and team plans** for secure, scalable adoption

Their platform is widely used by the ML community and features **over 2M models**.

## Community and culture

Hugging Face positions itself first and foremost as **a community**. Its public messaging emphasizes collaboration, openness, and shared progress in AI. The brand description highlights:

- **Open-source machine learning**
- **Learning and experimentation**
- **Collaboration between engineers, scientists, and end users**
- A commitment to **ethical AI**

The company is active across its **blog, forums, GitHub, and changelog**, reflecting a fast-moving, community-driven culture.

## Customers and users

Hugging Face serves a wide range of users, including:

- **Machine learning engineers**
- **Researchers and scientists**
- **AI builders and developers**
- **Organizations and enterprises**
- **End users exploring AI apps**

For businesses, Hugging Face offers enterprise features such as:

- **Single Sign-On**
- **Region controls for repository data**
- **Audit logs**
- **Priority support**

## Careers

Hugging Face is hiring and shares openings on its **careers page**. Its LinkedIn profile lists the company as:

- **Founded in 2016**
- **Privately held**
- **51–200 employees**
- Based in **NYC and Paris**, with a global presence

This suggests a relatively small but influential team working on core infrastructure for the AI ecosystem.

## Why it stands out

Hugging Face has become one of the most important platforms in modern AI because it combines:

- **A massive open model community**
- **Developer-friendly tooling**
- **Enterprise-ready services**
- **A strong open-source and ethical AI identity**

If you’re looking for a company shaping how the AI community builds and shares technology, Hugging Face is a standout name.

In [18]:
create_brochure("Ollama", "https://ollama.com/")


Selecting relevant links for https://ollama.com/ ...
Found 0 relevant links!


# Ollama

Ollama helps people and teams build with open models more easily. Its platform is designed to make it simple to get started locally, and then scale to cloud-based access when you need faster or larger models.

## What Ollama does

Ollama positions itself as “the easiest way to build with open models.” The company supports running models and launching AI-powered apps and agents quickly, including tools like Claude Code, Codex, and OpenClaw.

## Why customers use it

- Fast setup for working with open models
- Local-first workflow for getting started quickly
- Cloud options for larger and faster model access
- Support for running apps and agents powered by open models

## Product experience

Ollama emphasizes a straightforward developer experience:

- Install and get started quickly
- Run a model locally
- Launch agentic tools and apps from the Ollama environment
- Move from local development to cloud scaling as needed

## Company culture

From the messaging on the site, Ollama appears to value:
- Simplicity and ease of use
- Open model adoption
- Practical tooling for builders and developers
- Speed from experimentation to deployment

## Careers and jobs

No specific careers or hiring information was provided in the source content.

## For prospective customers

If you want to experiment with open models, build AI apps, or prototype agents without a complex setup, Ollama appears designed to make that process easier.

## For investors

Ollama is focused on the growing market for open-model tooling and deployment, with a product strategy that spans both local development and cloud scale.

## For recruits

Ollama appears to be a company centered on developer experience, AI infrastructure, and open model tooling—good fit areas for people interested in practical AI products and modern model workflows.